In [2]:
from verl.experimental.agent_loop.agent_loop import (
    AgentLoopBase,
    AgentLoopMetrics,
    AgentLoopOutput,
)

- 交互环境
- 动作空间设计
    - $\mathcal A_{\text{env}}=\mathcal A_{\text{bash}}\cup\mathcal A_{\text{editor}}\cup\{\texttt{submit}\}$
        - $\mathcal A_{\text{bash}}=\{\text{sandbox 中能够被 shell 解释的命令字符串}\}$
- reward fn：turns 的 reward/penalty
    -  tool-use shaping

- VERL AgentLoop → ModelProxy → sweagent 子进程 → SWE-ReX Docker sandbox → 命令执行/观察返回 → patch/reward

```mermaid
sequenceDiagram
    participant V as "VERL AgentLoop + vLLM (训练容器/GPU)"
    participant P as "ModelProxy (127.0.0.1:随机端口)"
    participant A as "sweagent 子进程 (训练容器)"
    participant R as "SWE-ReX"
    participant S as "Docker sandbox (CPU)"

    V->>P: 启动 OpenAI-compatible HTTP proxy
    V->>A: 启动 sweagent run
    A->>R: 根据 YAML 创建 Docker deployment
    R->>S: 启动容器、准备 repo 和工具

    A->>P: POST /v1/chat/completions
    P->>V: messages 转换为 token，交给 vLLM
    V->>P: 返回真实生成 token/text
    P->>A: OpenAI 格式 assistant response

    A->>R: 解析出 ls/cat/editor/pytest 等动作
    R->>S: 在同一个有状态 shell 中执行
    S->>R: stdout/stderr/exit code
    R->>A: observation

    A->>P: 带 observation 发起下一轮模型请求
    A->>S: submit
    S-->>A: 生成 model.patch
    A-->>V: patch + 多轮 trajectory
```